# Connect 4 AI Agent - Assignment 2

**Course:** CCE Artificial Intelligence  
**Topic:** Minimax, Alpha-Beta Pruning, and Expected Minimax  
**Implementation phase:** Notebook logic first, Streamlit GUI later.

This notebook focuses on the assignment core before the GUI: board representation, full-game scoring, heuristic evaluation, Minimax, Alpha-Beta pruning, Expected Minimax, tree tracing, notebook/console gameplay helpers, and comparison of time/nodes at different `K` values.


## 1. Assignment Requirements Mapped to AI Concepts

In this implementation:

- **Game state / node:** the current Connect 4 board.
- **Action / branch:** choosing a column to drop a disc.
- **MAX player:** the computer AI agent.
- **MIN player:** the human player.
- **Terminal state:** the board is full.
- **Utility / final score:** number of computer connected-fours minus number of human connected-fours.
- **K:** cutoff depth. The search stops after `K` plies and applies the heuristic function.
- **Heuristic value:** positive means the computer is in a better position; negative means the human is in a better position.

The assignment requires three AI options:

1. Minimax without Alpha-Beta pruning.
2. Minimax with Alpha-Beta pruning.
3. Expected Minimax with stochastic disc falling.


## 2. Important Design Notes

The assignment says the game continues until the board is full, and the winner is the player with the greater number of connected-fours. Therefore, this notebook **does not stop the game after the first connect-four**.

For search evaluation:

- Full terminal boards use the true assignment utility: computer connected-fours minus human connected-fours.
- Non-terminal cutoff boards at `depth == 0` use the heuristic function.
- The heuristic is scaled so connected-fours dominate smaller positional factors.

For Expected Minimax, the stochastic movement is handled as follows:

- `0.6` probability that the disc falls in the selected column.
- `0.2` probability that the disc falls in the left column.
- `0.2` probability that the disc falls in the right column.
- If a stochastic target column is outside the board or full, it is removed and the remaining probabilities are normalized.

Tree printing is implemented as a readable console trace. Every AI search can print the tree and leaf board values in a form that can be traced manually.


## 3. Imports and Constants


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from time import perf_counter
from typing import List, Tuple, Optional, Dict, Sequence
import math
import copy
import random
import pandas as pd


In [ ]:
ROWS = 6
COLS = 7

EMPTY = 0
HUMAN = 1       # MIN player
COMPUTER = 2    # MAX player

PLAYER_NAMES = {
    EMPTY: "EMPTY",
    HUMAN: "HUMAN",
    COMPUTER: "COMPUTER",
}

SYMBOLS = {
    EMPTY: ".",
    HUMAN: "H",
    COMPUTER: "C",
}

CENTER_FIRST_ORDER = [3, 2, 4, 1, 5, 0, 6]


## 4. Data Structures

The board is represented as a 2D list with `6` rows and `7` columns.

- `0` means empty.
- `1` means human disc.
- `2` means computer disc.

The bottom of the board is row index `5`, and discs fall down to the lowest available cell in a selected column.


In [ ]:
@dataclass
class SearchStats:
    nodes_expanded: int = 0
    leaf_evaluations: int = 0
    pruned_branches: int = 0
    max_depth_reached: int = 0


@dataclass
class SearchResult:
    algorithm: str
    best_column: Optional[int]
    value: float
    nodes_expanded: int
    leaf_evaluations: int
    pruned_branches: int
    running_time: float
    tree_trace: List[str] = field(default_factory=list)


## 5. Basic Board Functions


In [ ]:
def create_board() -> List[List[int]]:
    return [[EMPTY for _ in range(COLS)] for _ in range(ROWS)]


def copy_board(board: List[List[int]]) -> List[List[int]]:
    return [row[:] for row in board]


def is_valid_column(board: List[List[int]], col: int) -> bool:
    return 0 <= col < COLS and board[0][col] == EMPTY


def get_valid_columns(board: List[List[int]]) -> List[int]:
    return [col for col in range(COLS) if is_valid_column(board, col)]


def order_columns(valid_columns: Sequence[int]) -> List[int]:
    valid_set = set(valid_columns)
    return [col for col in CENTER_FIRST_ORDER if col in valid_set]


def get_next_open_row(board: List[List[int]], col: int) -> Optional[int]:
    for row in range(ROWS - 1, -1, -1):
        if board[row][col] == EMPTY:
            return row
    return None


def drop_piece(board: List[List[int]], col: int, piece: int) -> bool:
    row = get_next_open_row(board, col)
    if row is None:
        return False
    board[row][col] = piece
    return True


def is_board_full(board: List[List[int]]) -> bool:
    return all(board[0][col] != EMPTY for col in range(COLS))


def print_board(board: List[List[int]]) -> None:
    print("  " + " ".join(str(c) for c in range(COLS)))
    print(" +" + "--" * COLS + "+")
    for row in board:
        print(" |" + " ".join(SYMBOLS[cell] for cell in row) + " |")
    print(" +" + "--" * COLS + "+")


## 6. Connected-Fours and Utility

The assignment winner is based on the number of connected-fours after the board is full.  
So the final utility is:

**utility = computer connected-fours − human connected-fours**

A connected-four can be horizontal, vertical, diagonal down-right, or diagonal up-right.


In [ ]:
def all_windows(board: List[List[int]]) -> List[List[int]]:
    """Return every length-4 window on the board."""
    windows: List[List[int]] = []

    # Horizontal windows
    for row in range(ROWS):
        for col in range(COLS - 3):
            windows.append([board[row][col + i] for i in range(4)])

    # Vertical windows
    for row in range(ROWS - 3):
        for col in range(COLS):
            windows.append([board[row + i][col] for i in range(4)])

    # Diagonal down-right windows
    for row in range(ROWS - 3):
        for col in range(COLS - 3):
            windows.append([board[row + i][col + i] for i in range(4)])

    # Diagonal up-right windows
    for row in range(3, ROWS):
        for col in range(COLS - 3):
            windows.append([board[row - i][col + i] for i in range(4)])

    return windows


def count_connected_fours(board: List[List[int]], piece: int) -> int:
    return sum(1 for window in all_windows(board) if window.count(piece) == 4)


def utility(board: List[List[int]]) -> int:
    computer_fours = count_connected_fours(board, COMPUTER)
    human_fours = count_connected_fours(board, HUMAN)
    return computer_fours - human_fours


def current_scores(board: List[List[int]]) -> Tuple[int, int]:
    return count_connected_fours(board, COMPUTER), count_connected_fours(board, HUMAN)


## 7. Heuristic Functions

Because the full game tree is too large, the search stops after `K` levels and evaluates non-terminal cutoff boards using a heuristic function.

This notebook includes two heuristic functions:

1. `heuristic` - the main multi-factor heuristic used by default.
2. `heuristic_v2` - a simpler bonus/alternative heuristic used for comparison.

Both heuristics follow the same direction:

- Large positive values mean the computer is closer to winning.
- Large negative values mean the human is closer to winning.
- Completed connected-fours are worth `?100000`, so they dominate smaller positional factors.

The main heuristic considers:

- Current connected-fours.
- Windows with 3 discs and 1 empty cell.
- Windows with 2 discs and 2 empty cells.
- Center-column control.
- Blocking human threats.

The bonus `heuristic_v2` is intentionally simpler: it scores completed fours, immediate 3-in-a-row threats, and center control only. This makes it useful for showing how different heuristic designs affect the selected move.


In [ ]:
def evaluate_window(window: List[int]) -> int:
    """Evaluate one 4-cell window from the computer's point of view."""
    score = 0

    computer_count = window.count(COMPUTER)
    human_count = window.count(HUMAN)
    empty_count = window.count(EMPTY)

    # 3-in-a-row threats. Human penalty (-150) intentionally exceeds computer
    # bonus (+120): blocking an opponent's three is more urgent than extending
    # our own. Same asymmetric logic applies to the 2-in-a-row weights below.
    if computer_count == 3 and empty_count == 1:
        score += 120
    elif human_count == 3 and empty_count == 1:
        score -= 150

    # Medium opportunities
    elif computer_count == 2 and empty_count == 2:
        score += 15
    elif human_count == 2 and empty_count == 2:
        score -= 20

    # Small positional potential
    elif computer_count == 1 and empty_count == 3:
        score += 2
    elif human_count == 1 and empty_count == 3:
        score -= 2

    return score


def center_control_score(board: List[List[int]]) -> int:
    center_col = COLS // 2
    center_cells = [board[row][center_col] for row in range(ROWS)]
    return 6 * center_cells.count(COMPUTER) - 6 * center_cells.count(HUMAN)


def heuristic(board: List[List[int]]) -> int:
    """Main multi-factor heuristic used by default."""
    score = 0

    # Current connected-fours are highly important because the final winner depends on their count.
    computer_fours, human_fours = current_scores(board)
    score += 100000 * (computer_fours - human_fours)

    # Evaluate every length-4 candidate window.
    for window in all_windows(board):
        score += evaluate_window(window)

    # Prefer center control because it participates in many possible lines.
    score += center_control_score(board)

    return score


### 7.1 Bonus Heuristic

`heuristic_v2` is placed here beside the main heuristic so the two evaluation designs are easy to compare. It is intentionally simpler: completed fours, immediate 3-in-a-row threats, and center control only.


In [ ]:
def heuristic_v2(board: List[List[int]]) -> int:
    """Bonus heuristic: completed fours + immediate 3-threats only."""
    score = 0
    for window in all_windows(board):
        c = window.count(COMPUTER)
        h = window.count(HUMAN)
        e = window.count(EMPTY)
        if c == 4:
            score += 100000
        elif h == 4:
            score -= 100000
        elif c == 3 and e == 1:
            score += 50
        elif h == 3 and e == 1:
            score -= 80
    score += center_control_score(board)
    return score


### 7.2 Search Leaf Evaluation

Terminal full boards use the real assignment utility. Non-terminal cutoff leaves use the selected heuristic function.


In [ ]:
_active_heuristic = heuristic  # overridden by choose_ai_move at runtime
TERMINAL_UTILITY_SCALE = 100000


def evaluate_search_leaf(board: List[List[int]]) -> Tuple[float, str]:
    """Return the correct value for a search leaf and a label for tracing.

    Terminal full boards use the real assignment utility. Cutoff leaves use the
    selected heuristic, because their full-game result is not known yet.
    """
    if is_board_full(board):
        return TERMINAL_UTILITY_SCALE * utility(board), "terminal_utility"
    return _active_heuristic(board), "heuristic"


## 8. Minimax without Alpha-Beta Pruning

This follows the lecture style:

- `MAXIMIZE` searches for the child state with the highest utility.
- `MINIMIZE` searches for the child state with the lowest utility.
- At `depth == 0` or a full board, the board is evaluated using the heuristic function.


In [ ]:
def add_trace(lines: List[str], text: str, max_lines: int = 2000) -> None:
    if len(lines) < max_lines - 1:
        lines.append(text)
    elif len(lines) == max_lines - 1:
        lines.append("[tree trace truncated; search still continued]")


def maximize_plain(
    board: List[List[int]],
    depth: int,
    stats: SearchStats,
    lines: List[str],
    level: int = 0,
    max_trace_depth: int = 4,
) -> Tuple[Optional[int], float]:
    stats.nodes_expanded += 1
    stats.max_depth_reached = max(stats.max_depth_reached, level)
    indent = "  " * level

    if depth == 0 or is_board_full(board):
        value, evaluation_name = evaluate_search_leaf(board)
        stats.leaf_evaluations += 1
        if level <= max_trace_depth:
            board_rows = [indent + "  " + " ".join(SYMBOLS[cell] for cell in row) for row in board]
            nl = chr(10)
            leaf_msg = indent + "LEAF board:" + nl + nl.join(board_rows) + nl + indent + evaluation_name + "=" + str(value)
            add_trace(lines, leaf_msg)
        return None, value

    valid_columns = order_columns(get_valid_columns(board))
    if not valid_columns:
        value, evaluation_name = evaluate_search_leaf(board)
        stats.leaf_evaluations += 1
        return None, value

    max_child = valid_columns[0]
    max_utility = -math.inf

    if level <= max_trace_depth:
        add_trace(lines, f"{indent}MAX depth={depth}")

    for child_col in valid_columns:
        child_board = copy_board(board)
        drop_piece(child_board, child_col, COMPUTER)

        _, utility_value = minimize_plain(
            child_board, depth - 1, stats, lines, level + 1, max_trace_depth
        )

        if level <= max_trace_depth:
            add_trace(lines, f"{indent}  column {child_col} -> value {utility_value}")

        if utility_value > max_utility:
            max_child = child_col
            max_utility = utility_value

    if level <= max_trace_depth:
        add_trace(lines, f"{indent}MAX returns column={max_child}, value={max_utility}")

    return max_child, max_utility


def minimize_plain(
    board: List[List[int]],
    depth: int,
    stats: SearchStats,
    lines: List[str],
    level: int = 0,
    max_trace_depth: int = 4,
) -> Tuple[Optional[int], float]:
    stats.nodes_expanded += 1
    stats.max_depth_reached = max(stats.max_depth_reached, level)
    indent = "  " * level

    if depth == 0 or is_board_full(board):
        value, evaluation_name = evaluate_search_leaf(board)
        stats.leaf_evaluations += 1
        if level <= max_trace_depth:
            board_rows = [indent + "  " + " ".join(SYMBOLS[cell] for cell in row) for row in board]
            nl = chr(10)
            leaf_msg = indent + "LEAF board:" + nl + nl.join(board_rows) + nl + indent + evaluation_name + "=" + str(value)
            add_trace(lines, leaf_msg)
        return None, value

    valid_columns = order_columns(get_valid_columns(board))
    if not valid_columns:
        value, evaluation_name = evaluate_search_leaf(board)
        stats.leaf_evaluations += 1
        return None, value

    min_child = valid_columns[0]
    min_utility = math.inf

    if level <= max_trace_depth:
        add_trace(lines, f"{indent}MIN depth={depth}")

    for child_col in valid_columns:
        child_board = copy_board(board)
        drop_piece(child_board, child_col, HUMAN)

        _, utility_value = maximize_plain(
            child_board, depth - 1, stats, lines, level + 1, max_trace_depth
        )

        if level <= max_trace_depth:
            add_trace(lines, f"{indent}  column {child_col} -> value {utility_value}")

        if utility_value < min_utility:
            min_child = child_col
            min_utility = utility_value

    if level <= max_trace_depth:
        add_trace(lines, f"{indent}MIN returns column={min_child}, value={min_utility}")

    return min_child, min_utility


## 9. Minimax with Alpha-Beta Pruning

Alpha-Beta gives the same final decision as normal Minimax, but it avoids exploring branches that cannot affect the final decision.

- `alpha`: best value MAX can guarantee so far.
- `beta`: best value MIN can guarantee so far.
- If `alpha >= beta`, the remaining children are pruned.


In [ ]:
def maximize_ab(
    board: List[List[int]],
    depth: int,
    alpha: float,
    beta: float,
    stats: SearchStats,
    lines: List[str],
    level: int = 0,
    max_trace_depth: int = 4,
) -> Tuple[Optional[int], float]:
    stats.nodes_expanded += 1
    stats.max_depth_reached = max(stats.max_depth_reached, level)
    indent = "  " * level

    if depth == 0 or is_board_full(board):
        value, evaluation_name = evaluate_search_leaf(board)
        stats.leaf_evaluations += 1
        if level <= max_trace_depth:
            board_rows = [indent + "  " + " ".join(SYMBOLS[cell] for cell in row) for row in board]
            nl = chr(10)
            leaf_msg = indent + "LEAF board:" + nl + nl.join(board_rows) + nl + indent + evaluation_name + "=" + str(value)
            add_trace(lines, leaf_msg)
        return None, value

    valid_columns = order_columns(get_valid_columns(board))
    if not valid_columns:
        value, evaluation_name = evaluate_search_leaf(board)
        stats.leaf_evaluations += 1
        return None, value

    max_child = valid_columns[0]
    max_utility = -math.inf

    if level <= max_trace_depth:
        add_trace(lines, f"{indent}MAX depth={depth}, alpha={alpha}, beta={beta}")

    for index, child_col in enumerate(valid_columns):
        child_board = copy_board(board)
        drop_piece(child_board, child_col, COMPUTER)

        _, utility_value = minimize_ab(
            child_board, depth - 1, alpha, beta, stats, lines, level + 1, max_trace_depth
        )

        if utility_value > max_utility:
            max_child = child_col
            max_utility = utility_value

        alpha = max(alpha, max_utility)

        if level <= max_trace_depth:
            add_trace(
                lines,
                f"{indent}  column {child_col} -> value {utility_value}, alpha={alpha}, beta={beta}",
            )

        if alpha >= beta:
            remaining = len(valid_columns) - index - 1
            stats.pruned_branches += remaining
            if level <= max_trace_depth:
                add_trace(lines, f"{indent}  PRUNE remaining {remaining} branch(es)")
            break

    if level <= max_trace_depth:
        add_trace(lines, f"{indent}MAX returns column={max_child}, value={max_utility}")

    return max_child, max_utility


def minimize_ab(
    board: List[List[int]],
    depth: int,
    alpha: float,
    beta: float,
    stats: SearchStats,
    lines: List[str],
    level: int = 0,
    max_trace_depth: int = 4,
) -> Tuple[Optional[int], float]:
    stats.nodes_expanded += 1
    stats.max_depth_reached = max(stats.max_depth_reached, level)
    indent = "  " * level

    if depth == 0 or is_board_full(board):
        value, evaluation_name = evaluate_search_leaf(board)
        stats.leaf_evaluations += 1
        if level <= max_trace_depth:
            board_rows = [indent + "  " + " ".join(SYMBOLS[cell] for cell in row) for row in board]
            nl = chr(10)
            leaf_msg = indent + "LEAF board:" + nl + nl.join(board_rows) + nl + indent + evaluation_name + "=" + str(value)
            add_trace(lines, leaf_msg)
        return None, value

    valid_columns = order_columns(get_valid_columns(board))
    if not valid_columns:
        value, evaluation_name = evaluate_search_leaf(board)
        stats.leaf_evaluations += 1
        return None, value

    min_child = valid_columns[0]
    min_utility = math.inf

    if level <= max_trace_depth:
        add_trace(lines, f"{indent}MIN depth={depth}, alpha={alpha}, beta={beta}")

    for index, child_col in enumerate(valid_columns):
        child_board = copy_board(board)
        drop_piece(child_board, child_col, HUMAN)

        _, utility_value = maximize_ab(
            child_board, depth - 1, alpha, beta, stats, lines, level + 1, max_trace_depth
        )

        if utility_value < min_utility:
            min_child = child_col
            min_utility = utility_value

        beta = min(beta, min_utility)

        if level <= max_trace_depth:
            add_trace(
                lines,
                f"{indent}  column {child_col} -> value {utility_value}, alpha={alpha}, beta={beta}",
            )

        if alpha >= beta:
            remaining = len(valid_columns) - index - 1
            stats.pruned_branches += remaining
            if level <= max_trace_depth:
                add_trace(lines, f"{indent}  PRUNE remaining {remaining} branch(es)")
            break

    if level <= max_trace_depth:
        add_trace(lines, f"{indent}MIN returns column={min_child}, value={min_utility}")

    return min_child, min_utility


## 10. Expected Minimax

Expected Minimax is used when there is randomness.  
Here, the intended column may not be the actual column where the disc falls.

For each intended action:

- `0.6` goes to the selected column.
- `0.2` goes to the left column.
- `0.2` goes to the right column.

The expected value is the weighted sum of the possible resulting states.

The assignment specifies '0.6 chosen, 0.4 left or right', which we interpret as the standard 0.6 / 0.2 / 0.2 split.


In [ ]:
def chance_outcomes(board: List[List[int]], intended_col: int) -> List[Tuple[int, float]]:
    """Return valid actual columns and normalized probabilities for stochastic falling."""
    raw_outcomes = [
        (intended_col, 0.6),
        (intended_col - 1, 0.2),
        (intended_col + 1, 0.2),
    ]

    valid_outcomes: List[Tuple[int, float]] = []
    for actual_col, probability in raw_outcomes:
        if is_valid_column(board, actual_col):
            valid_outcomes.append((actual_col, probability))

    total_probability = sum(prob for _, prob in valid_outcomes)
    if total_probability == 0:
        return []

    return [(col, prob / total_probability) for col, prob in valid_outcomes]


def choose_stochastic_actual_column(
    board: List[List[int]],
    intended_col: int,
    rng: Optional[random.Random] = None,
) -> Optional[int]:
    """Sample the actual column for Expected Minimax gameplay."""
    outcomes = chance_outcomes(board, intended_col)
    if not outcomes:
        return None

    generator = rng if rng is not None else random
    threshold = generator.random()
    cumulative_probability = 0.0

    for actual_col, probability in outcomes:
        cumulative_probability += probability
        if threshold <= cumulative_probability:
            return actual_col

    return outcomes[-1][0]


def apply_game_move(
    board: List[List[int]],
    intended_col: int,
    piece: int,
    stochastic: bool = False,
    rng: Optional[random.Random] = None,
) -> Optional[int]:
    """Apply a deterministic or stochastic move and return the actual column used."""
    if stochastic:
        actual_col = choose_stochastic_actual_column(board, intended_col, rng=rng)
    else:
        actual_col = intended_col if is_valid_column(board, intended_col) else None

    if actual_col is None:
        return None

    drop_piece(board, actual_col, piece)
    return actual_col


def expected_maximize(
    board: List[List[int]],
    depth: int,
    stats: SearchStats,
    lines: List[str],
    level: int = 0,
    max_trace_depth: int = 3,
) -> Tuple[Optional[int], float]:
    stats.nodes_expanded += 1
    stats.max_depth_reached = max(stats.max_depth_reached, level)
    indent = "  " * level

    if depth == 0 or is_board_full(board):
        value, evaluation_name = evaluate_search_leaf(board)
        stats.leaf_evaluations += 1
        if level <= max_trace_depth:
            board_rows = [indent + "  " + " ".join(SYMBOLS[cell] for cell in row) for row in board]
            nl = chr(10)
            leaf_msg = indent + "LEAF board:" + nl + nl.join(board_rows) + nl + indent + evaluation_name + "=" + str(value)
            add_trace(lines, leaf_msg)
        return None, value

    valid_columns = order_columns(get_valid_columns(board))
    if not valid_columns:
        value, evaluation_name = evaluate_search_leaf(board)
        stats.leaf_evaluations += 1
        return None, value

    best_col = valid_columns[0]
    best_expected_value = -math.inf

    if level <= max_trace_depth:
        add_trace(lines, f"{indent}EXPECTED MAX depth={depth}")

    for intended_col in valid_columns:
        expected_value = 0.0
        outcomes = chance_outcomes(board, intended_col)

        for actual_col, probability in outcomes:
            child_board = copy_board(board)
            drop_piece(child_board, actual_col, COMPUTER)
            _, child_value = expected_minimize(
                child_board, depth - 1, stats, lines, level + 1, max_trace_depth
            )
            expected_value += probability * child_value
            if level <= max_trace_depth:
                add_trace(
                    lines,
                    f"{indent}    actual {actual_col} with p={probability:.2f} -> value {child_value:.2f}",
                )

        if level <= max_trace_depth:
            add_trace(lines, f"{indent}  intended {intended_col} -> expected value {expected_value:.2f}")

        if expected_value > best_expected_value:
            best_expected_value = expected_value
            best_col = intended_col

    if level <= max_trace_depth:
        add_trace(lines, f"{indent}EXPECTED MAX returns column={best_col}, value={best_expected_value:.2f}")

    return best_col, best_expected_value


def expected_minimize(
    board: List[List[int]],
    depth: int,
    stats: SearchStats,
    lines: List[str],
    level: int = 0,
    max_trace_depth: int = 3,
) -> Tuple[Optional[int], float]:
    stats.nodes_expanded += 1
    stats.max_depth_reached = max(stats.max_depth_reached, level)
    indent = "  " * level

    if depth == 0 or is_board_full(board):
        value, evaluation_name = evaluate_search_leaf(board)
        stats.leaf_evaluations += 1
        if level <= max_trace_depth:
            board_rows = [indent + "  " + " ".join(SYMBOLS[cell] for cell in row) for row in board]
            nl = chr(10)
            leaf_msg = indent + "LEAF board:" + nl + nl.join(board_rows) + nl + indent + evaluation_name + "=" + str(value)
            add_trace(lines, leaf_msg)
        return None, value

    valid_columns = order_columns(get_valid_columns(board))
    if not valid_columns:
        value, evaluation_name = evaluate_search_leaf(board)
        stats.leaf_evaluations += 1
        return None, value

    best_col = valid_columns[0]
    best_expected_value = math.inf

    if level <= max_trace_depth:
        add_trace(lines, f"{indent}EXPECTED MIN depth={depth}")

    for intended_col in valid_columns:
        expected_value = 0.0
        outcomes = chance_outcomes(board, intended_col)

        for actual_col, probability in outcomes:
            child_board = copy_board(board)
            drop_piece(child_board, actual_col, HUMAN)
            _, child_value = expected_maximize(
                child_board, depth - 1, stats, lines, level + 1, max_trace_depth
            )
            expected_value += probability * child_value
            if level <= max_trace_depth:
                add_trace(
                    lines,
                    f"{indent}    actual {actual_col} with p={probability:.2f} -> value {child_value:.2f}",
                )

        if level <= max_trace_depth:
            add_trace(lines, f"{indent}  intended {intended_col} -> expected value {expected_value:.2f}")

        if expected_value < best_expected_value:
            best_expected_value = expected_value
            best_col = intended_col

    if level <= max_trace_depth:
        add_trace(lines, f"{indent}EXPECTED MIN returns column={best_col}, value={best_expected_value:.2f}")

    return best_col, best_expected_value


## 11. AI Move Selection and Move Application

`choose_ai_move` is the main search entry point used by the notebook examples and the GUI. `apply_game_move` is used when the selected column needs to become an actual dropped disc, including the stochastic behavior in Expected Minimax mode.


In [ ]:
def choose_ai_move(
    board: List[List[int]],
    algorithm: str,
    k: int,
    trace: bool = True,
    max_trace_depth: int = 4,
    heuristic_fn=None,
) -> SearchResult:
    global _active_heuristic
    if k < 1:
        raise ValueError("K must be at least 1 because the AI must search at least one move.")

    _active_heuristic = heuristic_fn if heuristic_fn is not None else heuristic
    algorithm = algorithm.lower().strip()
    stats = SearchStats()
    lines: List[str] = []
    effective_trace_depth = max_trace_depth if trace else -1

    start_time = perf_counter()

    if algorithm in {"minimax", "plain", "normal"}:
        best_col, value = maximize_plain(board, k, stats, lines, max_trace_depth=effective_trace_depth)
        algorithm_name = "Minimax"

    elif algorithm in {"alpha-beta", "alphabeta", "ab"}:
        best_col, value = maximize_ab(
            board, k, -math.inf, math.inf, stats, lines, max_trace_depth=effective_trace_depth
        )
        algorithm_name = "Alpha-Beta"

    elif algorithm in {"expected", "expected-minimax", "expectiminimax"}:
        best_col, value = expected_maximize(board, k, stats, lines, max_trace_depth=effective_trace_depth)
        algorithm_name = "Expected Minimax"

    else:
        raise ValueError("Unknown algorithm. Use: minimax, alpha-beta, or expected-minimax.")

    running_time = perf_counter() - start_time

    return SearchResult(
        algorithm=algorithm_name,
        best_column=best_col,
        value=value,
        nodes_expanded=stats.nodes_expanded,
        leaf_evaluations=stats.leaf_evaluations,
        pruned_branches=stats.pruned_branches,
        running_time=running_time,
        tree_trace=lines,
    )


## 12. Notebook Gameplay Helpers

These helpers provide the non-GUI game loop while the Streamlit interface is still deferred. They use the same rules as the assignment: human vs computer, selected algorithm, selected `K`, tree printed on each AI turn, and final winner decided only after the board is full.


In [ ]:
def winner_summary(board: List[List[int]]) -> str:
    computer_score, human_score = current_scores(board)
    if computer_score > human_score:
        winner = "Computer wins"
    elif human_score > computer_score:
        winner = "Human wins"
    else:
        winner = "Draw"
    return f"{winner}: Computer {computer_score}, Human {human_score}"


def play_console_game(
    algorithm: str = "alpha-beta",
    k: int = 3,
    max_trace_depth: int = 3,
    heuristic_fn=None,
) -> List[List[int]]:
    """Play human vs computer in the notebook/console until the board is full."""
    board = create_board()
    stochastic = algorithm.lower().strip() in {"expected", "expected-minimax", "expectiminimax"}

    print_board(board)
    while not is_board_full(board):
        valid_columns = get_valid_columns(board)
        human_col = int(input(f"Human move {valid_columns}: "))
        if human_col not in valid_columns:
            print("Invalid column. Try again.")
            continue

        actual_human_col = apply_game_move(board, human_col, HUMAN, stochastic=stochastic)
        print(f"Human selected {human_col}; actual column {actual_human_col}")
        print_board(board)

        if is_board_full(board):
            break

        result = choose_ai_move(
            board,
            algorithm=algorithm,
            k=k,
            trace=True,
            max_trace_depth=max_trace_depth,
            heuristic_fn=heuristic_fn,
        )
        print("\nAI tree trace:")
        for line in result.tree_trace:
            print(line)

        actual_computer_col = apply_game_move(board, result.best_column, COMPUTER, stochastic=stochastic)
        print(f"Computer selected {result.best_column}; actual column {actual_computer_col}")
        print("Search value:", result.value)
        print("Nodes expanded:", result.nodes_expanded)
        print("Time:", result.running_time)
        print_board(board)

    print("Final result:", winner_summary(board))
    return board


def play_scripted_game(
    human_moves: Sequence[int],
    algorithm: str = "alpha-beta",
    k: int = 3,
    max_trace_depth: int = 2,
    stochastic_seed: int = 0,
    print_trees: bool = True,
    heuristic_fn=None,
) -> Tuple[List[List[int]], List[Dict[str, object]]]:
    """Run a reproducible non-GUI game from a list of human columns."""
    board = create_board()
    rng = random.Random(stochastic_seed)
    stochastic = algorithm.lower().strip() in {"expected", "expected-minimax", "expectiminimax"}
    move_log: List[Dict[str, object]] = []

    for turn_number, human_col in enumerate(human_moves, start=1):
        if is_board_full(board):
            break
        if not is_valid_column(board, human_col):
            raise ValueError(f"Invalid human move at turn {turn_number}: column {human_col}")

        actual_human_col = apply_game_move(board, human_col, HUMAN, stochastic=stochastic, rng=rng)
        move_log.append({"turn": turn_number, "player": "human", "intended": human_col, "actual": actual_human_col})

        if is_board_full(board):
            break

        result = choose_ai_move(
            board,
            algorithm=algorithm,
            k=k,
            trace=print_trees,
            max_trace_depth=max_trace_depth,
            heuristic_fn=heuristic_fn,
        )
        actual_computer_col = apply_game_move(board, result.best_column, COMPUTER, stochastic=stochastic, rng=rng)
        move_log.append({
            "turn": turn_number,
            "player": "computer",
            "intended": result.best_column,
            "actual": actual_computer_col,
            "value": result.value,
            "nodes": result.nodes_expanded,
            "time": result.running_time,
        })

        if print_trees:
            print(f"\nTurn {turn_number} AI tree trace:")
            for line in result.tree_trace:
                print(line)

    return board, move_log


# Example interactive call, left commented so running all cells does not pause for input:
# final_board = play_console_game(algorithm="alpha-beta", k=3)


## 13. Sample Board for Testing

This board is used to test all algorithms with the same state.


In [ ]:
sample_board = create_board()

# A small non-empty state for testing.
for col, piece in [
    (3, HUMAN),
    (3, COMPUTER),
    (2, HUMAN),
    (4, COMPUTER),
    (2, HUMAN),
    (4, COMPUTER),
    (1, HUMAN),
]:
    drop_piece(sample_board, col, piece)

print_board(sample_board)

computer_fours, human_fours = current_scores(sample_board)
print("Computer connected-fours:", computer_fours)
print("Human connected-fours:", human_fours)
print("Heuristic value:", heuristic(sample_board))


## 14. Test Minimax without Alpha-Beta


In [ ]:
K = 3

result_minimax = choose_ai_move(sample_board, algorithm="minimax", k=K, trace=True, max_trace_depth=3)

print("Algorithm:", result_minimax.algorithm)
print("Best column:", result_minimax.best_column)
print("Value:", result_minimax.value)
print("Nodes expanded:", result_minimax.nodes_expanded)
print("Leaf evaluations:", result_minimax.leaf_evaluations)
print("Time:", result_minimax.running_time)

print("\nTree trace:")
for line in result_minimax.tree_trace[:120]:
    print(line)

print("\n--- Board after AI move (col", result_minimax.best_column, ") ---")
post_board = copy_board(sample_board)
drop_piece(post_board, result_minimax.best_column, COMPUTER)
print_board(post_board)
print("Heuristic value of resulting board:", heuristic(post_board))


## 15. Test Minimax with Alpha-Beta Pruning


In [ ]:
K = 3

result_ab = choose_ai_move(sample_board, algorithm="alpha-beta", k=K, trace=True, max_trace_depth=3)

print("Algorithm:", result_ab.algorithm)
print("Best column:", result_ab.best_column)
print("Value:", result_ab.value)
print("Nodes expanded:", result_ab.nodes_expanded)
print("Leaf evaluations:", result_ab.leaf_evaluations)
print("Pruned branches:", result_ab.pruned_branches)
print("Time:", result_ab.running_time)

print("\nTree trace:")
for line in result_ab.tree_trace[:120]:
    print(line)

print("\n--- Board after AI move (col", result_ab.best_column, ") ---")
post_board_ab = copy_board(sample_board)
drop_piece(post_board_ab, result_ab.best_column, COMPUTER)
print_board(post_board_ab)
print("Heuristic value of resulting board:", heuristic(post_board_ab))


## 16. Test Expected Minimax


In [ ]:
K = 2

result_expected = choose_ai_move(sample_board, algorithm="expected-minimax", k=K, trace=True, max_trace_depth=2)

print("Algorithm:", result_expected.algorithm)
print("Best intended column:", result_expected.best_column)
print("Expected value:", result_expected.value)
print("Nodes expanded:", result_expected.nodes_expanded)
print("Leaf evaluations:", result_expected.leaf_evaluations)
print("Time:", result_expected.running_time)
print("Chance outcomes from best intended column:", chance_outcomes(sample_board, result_expected.best_column))

print("\nTree trace:")
for line in result_expected.tree_trace[:120]:
    print(line)

print("\n--- Board after stochastic AI move ---")
post_board_ex = copy_board(sample_board)
actual_expected_col = apply_game_move(
    post_board_ex,
    result_expected.best_column,
    COMPUTER,
    stochastic=True,
    rng=random.Random(7),
)
print("Intended column:", result_expected.best_column)
print("Actual column:", actual_expected_col)
print_board(post_board_ex)
print("Heuristic value of resulting board:", heuristic(post_board_ex))


## 17. Comparison at Different K Values

The report requires a comparison between normal Minimax and Alpha-Beta in terms of:

- Time taken.
- Nodes expanded.
- Different `K` values.


In [ ]:
def compare_minimax_and_alpha_beta(
    board: List[List[int]],
    k_values: Sequence[int] = (1, 2, 3, 4),
) -> pd.DataFrame:
    rows = []

    for k in k_values:
        for algorithm in ["minimax", "alpha-beta"]:
            result = choose_ai_move(board, algorithm=algorithm, k=k, trace=False)
            rows.append({
                "K": k,
                "Algorithm": result.algorithm,
                "Best Column": result.best_column,
                "Value": result.value,
                "Nodes Expanded": result.nodes_expanded,
                "Leaf Evaluations": result.leaf_evaluations,
                "Pruned Branches": result.pruned_branches,
                "Time (seconds)": result.running_time,
            })

    return pd.DataFrame(rows)


comparison_df = compare_minimax_and_alpha_beta(sample_board, k_values=(1, 2, 3, 4))
comparison_df


## 18. Report Assumptions and Requirement Checklist

Use these assumptions in the final report:

- The board size is fixed at **6 rows (length) x 7 columns (width)**, satisfying the assignment requirement: width >= 7 and length >= 6.
- The computer is the `MAX` player, and the human is the `MIN` player.
- The game ends only when the board is full.
- The winner is the player with the greater number of connected-fours.
- `K` is the cutoff search depth entered before search/gameplay.
- Full boards use the true utility: computer connected-fours minus human connected-fours.
- Non-terminal cutoff boards use the selected heuristic function.
- In Expected Minimax, the side probability `0.4` is split equally into `0.2` left and `0.2` right.
- If a stochastic target column is invalid or full, its probability is removed and the remaining probabilities are normalized.
- Center-first move ordering is used to make Alpha-Beta pruning more effective and consistent.
- The heuristic uses **asymmetric weights** (human threat penalty > computer bonus) deliberately: blocking an opponent's three-in-a-row is more urgent than extending our own.
- The tree trace is capped at 2,000 lines to keep console output manageable; at high `K` the trace is truncated but the algorithm still explores the full tree.

Requirement checklist for the notebook logic, excluding the GUI that will be built later in Streamlit:

- Human-vs-computer gameplay logic: implemented by `play_console_game` and `play_scripted_game`.
- Choose Alpha-Beta pruning or no pruning: provided by the `algorithm` parameter.
- Minimax, Alpha-Beta, and Expected Minimax options: implemented by `choose_ai_move`.
- Depth `K` parameter before search/gameplay: implemented by `choose_ai_move`, `play_console_game`, and `play_scripted_game`.
- Heuristic values shown: printed in examples and included in leaf board evaluations in the tree trace.
- Minimax tree printed each AI turn: implemented through each `SearchResult.tree_trace`; `play_console_game` prints it every AI turn.
- Timing and nodes-expanded comparison at different `K` values: generated in the comparison table below.
- Report material: assumptions, sample runs, tree traces, data structures, heuristic comparison, timing, and nodes expanded are all present in this notebook.

GUI status: intentionally deferred. The Streamlit app can call these notebook functions later without changing the AI logic.


## 19. Streamlit GUI Plan for Later

The GUI is intentionally left for a later Streamlit file. The backend functions above are ready for Streamlit:

- `create_board`, `drop_piece`, `apply_game_move` for board state updates.
- `choose_ai_move` for Minimax, Alpha-Beta, and Expected Minimax turns.
- `current_scores`, `heuristic`, and `winner_summary` for status display.
- `SearchResult.tree_trace` for showing the latest readable minimax tree in the app.

Required Streamlit controls later:

- Algorithm selector: Minimax, Alpha-Beta, or Expected Minimax.
- `K` input before starting the game.
- Column buttons for human moves.
- Current board display.
- Current connected-four scores and final winner after the board is full.
- Text area showing the readable tree trace for the last AI turn.


## 20. Additional Sample Runs

Three boards covering different game phases: **near-computer-win**, **near-human-win**, and **midgame**.


In [ ]:
# Board A: Computer has 3 in a row; col 4 is the winning move.
board_a = create_board()
for col, piece in [
    (0, HUMAN), (1, COMPUTER), (2, COMPUTER), (3, COMPUTER), (5, HUMAN),
    (0, COMPUTER),
]:
    drop_piece(board_a, col, piece)
print("Board A: Computer near-win")
print_board(board_a)
print("Heuristic:", heuristic(board_a))
ra = choose_ai_move(board_a, "alpha-beta", k=3, trace=True, max_trace_depth=3)
print("AI best column:", ra.best_column, "| value:", ra.value)
print("Nodes expanded:", ra.nodes_expanded, "| Pruned:", ra.pruned_branches)
print("Tree trace (first 60 lines):")
for ln in ra.tree_trace[:60]:
    print(ln)

# Board B: Human has one immediate winning square; AI should block col 6.
board_b = create_board()
for col, piece in [
    (2, COMPUTER),
    (3, HUMAN), (4, HUMAN), (5, HUMAN),
    (1, COMPUTER), (3, COMPUTER),
    (1, HUMAN),
]:
    drop_piece(board_b, col, piece)
print("\nBoard B: Human near-win with one open end (AI should block)")
print_board(board_b)
print("Heuristic:", heuristic(board_b))
rb = choose_ai_move(board_b, "alpha-beta", k=3, trace=True, max_trace_depth=3)
print("AI best column:", rb.best_column, "| value:", rb.value)
print("Nodes expanded:", rb.nodes_expanded, "| Pruned:", rb.pruned_branches)
print("Tree trace (first 60 lines):")
for ln in rb.tree_trace[:60]:
    print(ln)

# Board C: Midgame - scattered pieces, no immediate threat.
board_c = create_board()
for col, piece in [
    (3, HUMAN), (3, COMPUTER), (3, HUMAN),
    (2, COMPUTER), (2, HUMAN), (2, COMPUTER),
    (4, HUMAN), (4, COMPUTER),
    (1, COMPUTER), (5, HUMAN),
    (0, HUMAN), (6, COMPUTER),
]:
    drop_piece(board_c, col, piece)
print("\nBoard C: Midgame")
print_board(board_c)
print("Heuristic:", heuristic(board_c))
rc = choose_ai_move(board_c, "minimax", k=3, trace=True, max_trace_depth=3)
print("AI best column:", rc.best_column, "| value:", rc.value)
print("Nodes expanded:", rc.nodes_expanded)
print("Tree trace (first 60 lines):")
for ln in rc.tree_trace[:60]:
    print(ln)
